# Data and Knowledge Engineering Endprojekt: Players
- ### Name: Tobias Hundsberger
- ### Matrikelnummer: h12042646
---

**Hinweis:** Der verwendete Datensatz enthält Spieler aus dem Videospiel Fifa. Jeder Spieler im Spiel hat eine Gesamtbewertung, als auch Bewertung für einzelne Kategorien. Im Datensatz ist jedoch nur die Gesamtbewertung unter dem Punkt 'overall' enthalten.

In [1]:
import requests

#Lese csv Tabelle von meinem GitHub ein
url = 'https://raw.githubusercontent.com/TobiHb/Data-and-Knowledge-Engineering-Endprojekt/refs/heads/main/fifa_data.csv'
r = requests.get(url)
f = open('fifa.csv', 'wb')
f.write(r.content)
f.close()

In [2]:
#Lade die SQL extension
%load_ext sql

In [3]:
#Erstelle SQLite Datenbank mit dem Namen 'players'
%sql sqlite:///players.db

In [4]:
import sqlite3 
import pandas as pd
conn = sqlite3.connect('players.db') # connection to database in current directory

%sql drop table if exists players;
# read_csv: read data from file in current directory; to_sql: write data to database table
pd.read_csv(url).to_sql('players', conn, index = False) # no additional index column
%sql ALTER TABLE players ADD COLUMN Capital TEXT

 * sqlite:///players.db
Done.
 * sqlite:///players.db
Done.


[]

---
### Einen kurzen Überblick über die bisher erstellte Datebank:

In [5]:
%%sql
select * from players limit 5;

 * sqlite:///players.db
Done.


sofifa_id,player_url,short_name,long_name,age,dob,height_cm,weight_kg,nationality,club,overall,potential,value_eur,wage_eur,player_positions,preferred_foot,Capital
158023,https://sofifa.com/player/158023/lionel-messi/20/159586,L. Messi,Lionel Andrés Messi Cuccittini,32,1987-06-24,170,72,Argentina,FC Barcelona,94,94,95500000,565000,"RW, CF, ST",Left,None
20801,https://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/20/159586,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,34,1985-02-05,187,83,Portugal,Juventus,93,93,58500000,405000,"ST, LW",Right,None
190871,https://sofifa.com/player/190871/neymar-da-silva-santos-jr/20/159586,Neymar Jr,Neymar da Silva Santos Junior,27,1992-02-05,175,68,Brazil,Paris Saint-Germain,92,92,105500000,290000,"LW, CAM",Right,None
200389,https://sofifa.com/player/200389/jan-oblak/20/159586,J. Oblak,Jan Oblak,26,1993-01-07,188,87,Slovenia,Atlético Madrid,91,93,77500000,125000,GK,Right,None
183277,https://sofifa.com/player/183277/eden-hazard/20/159586,E. Hazard,Eden Hazard,28,1991-01-07,175,74,Belgium,Real Madrid,91,91,90000000,470000,"LW, CF",Right,None


--- 
### Über DBPedia werden neue Daten mit einer SparQL Abfrage abgerufen und der Datenbank 'players' hinzugefügt:

In [6]:
from SPARQLWrapper import SPARQLWrapper, JSON

# Erstelle Verbindung zur Datenbank
conn = sqlite3.connect('players.db')
cursor = conn.cursor()


sparql = SPARQLWrapper("http://dbpedia.org/sparql")

# Definiere Suchanfrage auf DBpedia Server
sparql.setQuery(f"""
    SELECT ?country ?capital
    WHERE {{
        ?country a <http://dbpedia.org/ontology/Country>;
                 <http://dbpedia.org/ontology/capital> ?capital.
        
        ?country rdfs:label ?country_label.
        ?capital rdfs:label ?capital_label.
        FILTER(LANG(?country_label) = "de" && LANG(?capital_label) = "de")
    }}
""")

# Lege Rückgabeformat fest
sparql.setReturnFormat(JSON)
# Weise Daten einer Variablen zu über welche später auf sie zugegriffen werden kann. 
capitals = sparql.query().convert()


# Überprüfe ob Suchanfrage Daten gefunden hat
if len(capitals['results']['bindings']) > 0:
    # Iteriere über Werte der Sparql Suchanfrage
    for result in capitals['results']['bindings']:
        # Formatiere die Werte, so dass sie in sauberer, schlichter, und lesbarer Form ausgelesen werden können.
        country = result["country"]["value"].split("/")[-1].replace("_", " ")
        capital = result["capital"]["value"].split("/")[-1].replace("_", " ")

        # Füge die Werte in die Datenbank players zu den passenden Ländern hinzu.
        cursor.execute("UPDATE players SET capital = ? WHERE nationality = ?", (capital, country))
        # Commit nach jedem Update
        conn.commit()  

    print("Hauptstädte erfolgreich hinzugefügt!")
else:
    print("Keine Hauptstädte gefunden.")

# Schließe die Verbindung zur Datenbank
conn.close()

Hauptstädte erfolgreich hinzugefügt!


---
### Überprüfe ob DBPedia Daten der Datenbank erfolgreich hinzugefügt worden sind:

In [7]:
%%sql
select nationality, capital from players limit 10;

 * sqlite:///players.db
Done.


nationality,Capital
Argentina,Buenos Aires
Portugal,Lisbon
Brazil,Brasília
Slovenia,Ljubljana
Belgium,City of Brussels
Belgium,City of Brussels
Germany,Berlin
Netherlands,None
Croatia,Zagreb
Egypt,Cairo


---
### Weiterer Test um zu überprüfen ob die SparQL Abfrage erfolgreich war:

In [8]:
for result in capitals["results"]["bindings"]:
    print("Country:", result["country"]["value"].split("/")[-1].replace("_", " "), "\n","Capital:", result["capital"]["value"].split("/")[-1].replace("_", " "))
    

Country: Roman Britain 
 Capital: Camulodunum
Country: Republic of Independent Guiana 
 Capital: Calçoene
Country: United Arab Republic 
 Capital: Cairo
Country: United Arab States 
 Capital: Cairo
Country: Saint-Domingue 
 Capital: Cap-Haïtien
Country: Egypt 
 Capital: Cairo
Country: Egypt Eyalet 
 Capital: Cairo
Country: Australia 
 Capital: Canberra
Country: Australia 
 Capital: Canberra
Country: Ayyubid dynasty 
 Capital: Cairo
Country: Abbasid Caliphate 
 Capital: Cairo
Country: Abbasid Caliphate 
 Capital: Cairo
Country: Fatimid Caliphate 
 Capital: Cairo
Country: Federation of Arab Republics 
 Capital: Cairo
Country: Khedivate of Egypt 
 Capital: Cairo
Country: Sultanate of Egypt 
 Capital: Cairo
Country: Syria Palaestina 
 Capital: Caesarea Maritima
Country: Syria Palaestina 
 Capital: Caesarea Maritima
Country: Burji dynasty 
 Capital: Cairo
Country: Kingdom of Egypt 
 Capital: Cairo
Country: Kingdom of Haiti 
 Capital: Cap-Haïtien
Country: Kingdom of Sardinia 
 Capital: Cagli

---
### Erstelle einen View aller brasilianischer Fussballspieler:

In [9]:
%%sql

drop view if exists BrazilNationalTeam;
create view BrazilNationalTeam as select short_name, club, player_positions, overall  from players where nationality == 'Brazil';
select * from BrazilNationalTeam;

 * sqlite:///players.db
Done.
Done.
Done.


short_name,club,player_positions,overall
Neymar Jr,Paris Saint-Germain,"LW, CAM",92
Alisson,Liverpool,GK,89
Ederson,Manchester City,GK,88
Casemiro,Real Madrid,CDM,87
Fernandinho,Manchester City,CDM,87
Thiago Silva,Paris Saint-Germain,CB,87
Marquinhos,Paris Saint-Germain,"CB, CDM",86
Roberto Firmino,Liverpool,"CF, ST, CAM",86
Coutinho,FC Bayern München,"LW, CM",86
Fabinho,Liverpool,CDM,85


---
### Wähle alle Spieler mit einer Gesamtbewertung von über 88:

In [10]:
%%sql
select short_name, club, player_positions, overall from players where overall > 88;

 * sqlite:///players.db
Done.


short_name,club,player_positions,overall
L. Messi,FC Barcelona,"RW, CF, ST",94
Cristiano Ronaldo,Juventus,"ST, LW",93
Neymar Jr,Paris Saint-Germain,"LW, CAM",92
J. Oblak,Atlético Madrid,GK,91
E. Hazard,Real Madrid,"LW, CF",91
K. De Bruyne,Manchester City,"CAM, CM",91
M. ter Stegen,FC Barcelona,GK,90
V. van Dijk,Liverpool,CB,90
L. Modrić,Real Madrid,CM,90
M. Salah,Liverpool,"RW, ST",90


---
### Sortiert die Spieler nach ihrem Gehalt:

In [11]:
%%sql
select short_name, long_name, club, nationality, capital, overall, wage_eur from players ORDER BY wage_eur DESC;

 * sqlite:///players.db
Done.


short_name,long_name,club,nationality,Capital,overall,wage_eur
L. Messi,Lionel Andrés Messi Cuccittini,FC Barcelona,Argentina,Buenos Aires,94,565000
E. Hazard,Eden Hazard,Real Madrid,Belgium,City of Brussels,91,470000
Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,Juventus,Portugal,Lisbon,93,405000
K. De Bruyne,Kevin De Bruyne,Manchester City,Belgium,City of Brussels,91,370000
A. Griezmann,Antoine Griezmann,FC Barcelona,France,Paris,89,370000
L. Suárez,Luis Alberto Suárez Díaz,FC Barcelona,Uruguay,Montevideo,89,355000
L. Modrić,Luka Modrić,Real Madrid,Croatia,Zagreb,90,340000
T. Kroos,Toni Kroos,Real Madrid,Germany,Berlin,88,330000
S. Agüero,Sergio Leonel Agüero del Castillo,Manchester City,Argentina,Buenos Aires,89,300000
Sergio Ramos,Sergio Ramos García,Real Madrid,Spain,Madrid,89,300000
